# Gold Layer End-to-End Audit (rerunnable)

Audits all 11 gold tables in `b2b_ecommerce.gold` (7 dims + 3 facts + ref_ip_geo):
1. **Key integrity** — rows = distinct keys, 0 duplicates, 0 null keys, SK = crc32 checks, SCD2 window validity
2. **FK integrity** — every fact SK joins to its dim (anti-join pattern; Photon-safe)
3. **Row deltas vs silver** — 0 rows lost/added per table; enrichment coverage
4. **KPI spot checks** — net revenue, cancelled %, bot share, lead conversion, geo split

Run all cells; each prints a check/value table. Expected values (verified 2026-11-19) are in the cell comments.

**Known-by-design (do not treat as failures):**
- `fact_order_items`: 1,074 lines with `order_date_valid=false` (474 null + 600 future order dates) carry NULL `date_sk` — flagged, not dropped.
- `fact_leads`: 954/1,098 leads have NULL `company_sk` (no company-name match — leads carry no company_id); 144 match, 140 of those have orders.
- `fact_web_logs`: 324,021 anonymous rows (NULL `customer_sk`), 9,698 NULL `user_agent` (surface as 'unknown' device), 176,174 NULL `referer`.
- Bots are KEPT in the fact (gross traffic, 140,414 rows / 26.1%); `dim_device.is_bot` is the view-side exclusion flag.

In [0]:
# 1. Key integrity: rows vs distinct keys, null keys, SK validity, SCD2 windows.
# Expected (2026-11-19): every 'rows' == its 'distinct' count; every null/dup/overlap/mismatch = 0.
import pandas as pd

checks = [
    ("dim_customers: rows", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_customers"),
    ("dim_customers: distinct customer_id / customer_sk", "SELECT CONCAT(COUNT(DISTINCT customer_id), '/', COUNT(DISTINCT customer_sk)) FROM b2b_ecommerce.gold.dim_customers"),
    ("dim_customers: null or wrong customer_sk", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_customers WHERE customer_sk IS NULL OR customer_sk < 0 OR customer_sk != crc32(CAST(customer_id AS STRING))"),
    ("dim_companies: SCD2 rows / distinct company_id / current", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT company_id), '/', SUM(CASE WHEN __END_AT IS NULL THEN 1 ELSE 0 END)) FROM b2b_ecommerce.gold.dim_companies"),
    ("dim_companies: keys with >1 current row", "SELECT COUNT(*) FROM (SELECT company_id FROM b2b_ecommerce.gold.dim_companies WHERE __END_AT IS NULL GROUP BY company_id HAVING COUNT(*) > 1)"),
    ("dim_companies: overlapping SCD2 windows", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_companies a JOIN b2b_ecommerce.gold.dim_companies b ON a.company_id = b.company_id AND a.__START_AT < b.__START_AT WHERE a.__END_AT IS NOT NULL AND a.__END_AT > b.__START_AT"),
    ("dim_products: SCD2 rows / distinct product_id / current", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT product_id), '/', SUM(CASE WHEN __END_AT IS NULL THEN 1 ELSE 0 END)) FROM b2b_ecommerce.gold.dim_products"),
    ("dim_products: keys with >1 current row", "SELECT COUNT(*) FROM (SELECT product_id FROM b2b_ecommerce.gold.dim_products WHERE __END_AT IS NULL GROUP BY product_id HAVING COUNT(*) > 1)"),
    ("dim_products: overlapping SCD2 windows", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_products a JOIN b2b_ecommerce.gold.dim_products b ON a.product_id = b.product_id AND a.__START_AT < b.__START_AT WHERE a.__END_AT IS NOT NULL AND a.__END_AT > b.__START_AT"),
    ("dim_suppliers: rows vs distinct supplier_id", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT supplier_id)) FROM b2b_ecommerce.gold.dim_suppliers"),
    ("dim_date: rows vs distinct date_sk", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT date_sk)) FROM b2b_ecommerce.gold.dim_date"),
    ("dim_device: rows / distinct UA / distinct SK", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT user_agent), '/', COUNT(DISTINCT device_sk)) FROM b2b_ecommerce.gold.dim_device"),
    ("dim_device: null or wrong device_sk", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_device WHERE device_sk IS NULL OR device_sk < 0 OR device_sk != crc32(CAST(user_agent AS STRING))"),
    ("dim_device: bad device_type / is_bot mismatches", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_device WHERE device_type NOT IN ('desktop','mobile','tablet','bot','other') OR is_bot != (device_type = 'bot')"),
    ("dim_ip_geo: rows / distinct IP / distinct SK", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT client_ip), '/', COUNT(DISTINCT ip_sk)) FROM b2b_ecommerce.gold.dim_ip_geo"),
    ("dim_ip_geo: null or wrong ip_sk", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_ip_geo WHERE ip_sk IS NULL OR ip_sk < 0 OR ip_sk != crc32(CAST(client_ip AS STRING))"),
    ("dim_ip_geo: country set with NULL octet (impossible)", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_ip_geo WHERE country_code IS NOT NULL AND first_octet IS NULL"),
    ("ref_ip_geo: duplicate first_octet", "SELECT COUNT(*) - COUNT(DISTINCT first_octet) FROM b2b_ecommerce.gold.ref_ip_geo"),
    ("fact_order_items: rows vs distinct order_item_id", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT order_item_id)) FROM b2b_ecommerce.gold.fact_order_items"),
    ("fact_leads: rows vs distinct lead_id", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT lead_id)) FROM b2b_ecommerce.gold.fact_leads"),
    ("fact_web_logs: rows vs distinct log_id", "SELECT CONCAT(COUNT(*), '/', COUNT(DISTINCT log_id)) FROM b2b_ecommerce.gold.fact_web_logs"),
]
rows = [(label, spark.sql(q).collect()[0][0]) for label, q in checks]
display(pd.DataFrame(rows, columns=["check", "value"]))

In [0]:
# 2. FK anti-joins (Photon-safe), row deltas vs silver, enrichment coverage, null profile.
# Expected (2026-11-19): every FK/orphan/missing/delta = 0. The known-by-design NULLs are labeled.
import pandas as pd

checks = [
    ("FK fact_order_items.customer_sk -> dim_customers", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items f LEFT ANTI JOIN b2b_ecommerce.gold.dim_customers d ON f.customer_sk = d.customer_sk"),
    ("FK fact_order_items.company_sk -> dim_companies", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items f LEFT ANTI JOIN b2b_ecommerce.gold.dim_companies d ON f.company_sk = d.company_sk"),
    ("FK fact_order_items.product_sk -> dim_products", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items f LEFT ANTI JOIN b2b_ecommerce.gold.dim_products d ON f.product_sk = d.product_sk"),
    ("FK fact_order_items.supplier_sk -> dim_suppliers", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items f LEFT ANTI JOIN b2b_ecommerce.gold.dim_suppliers d ON f.supplier_sk = d.supplier_sk"),
    ("FK fact_order_items.date_sk orphans with order_date_valid=true", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items f LEFT ANTI JOIN b2b_ecommerce.gold.dim_date d ON f.date_sk = d.date_sk WHERE f.order_date_valid = true"),
    ("FK fact_order_items: null date_sk (all order_date_valid=false)", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items WHERE date_sk IS NULL AND order_date_valid != false"),
    ("FK fact_leads.date_sk -> dim_date", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_leads f LEFT ANTI JOIN b2b_ecommerce.gold.dim_date d ON f.date_sk = d.date_sk"),
    ("FK fact_leads.company_sk orphans (non-null only)", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_leads f LEFT ANTI JOIN b2b_ecommerce.gold.dim_companies d ON f.company_sk = d.company_sk WHERE f.company_sk IS NOT NULL"),
    ("FK fact_web_logs.date_sk -> dim_date", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs f LEFT ANTI JOIN b2b_ecommerce.gold.dim_date d ON f.date_sk = d.date_sk"),
    ("FK fact_web_logs.customer_sk orphans (non-null only)", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs f LEFT ANTI JOIN b2b_ecommerce.gold.dim_customers d ON f.customer_sk = d.customer_sk WHERE f.customer_sk IS NOT NULL"),
    ("FK dim_customers.company_id -> dim_companies", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_customers c LEFT ANTI JOIN b2b_ecommerce.gold.dim_companies d ON c.company_id = d.company_id"),
    ("FK dim_suppliers.company_id -> dim_companies", "SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_suppliers s LEFT ANTI JOIN b2b_ecommerce.gold.dim_companies d ON s.company_id = d.company_id"),
    ("delta fact_order_items vs silver.order_items", "SELECT (SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_order_items) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.order_items)"),
    ("delta fact_order_items: silver rows missing in gold", "SELECT COUNT(*) FROM b2b_ecommerce.silver.order_items s LEFT ANTI JOIN b2b_ecommerce.gold.fact_order_items f ON s.order_item_id = f.order_item_id"),
    ("delta fact_leads vs silver.leads", "SELECT (SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_leads) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.leads)"),
    ("delta fact_web_logs vs silver.logs", "SELECT (SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.logs)"),
    ("delta fact_web_logs: silver rows missing in gold", "SELECT COUNT(*) FROM b2b_ecommerce.silver.logs s LEFT ANTI JOIN b2b_ecommerce.gold.fact_web_logs f ON s.log_id = f.log_id"),
    ("delta dim_customers / dim_companies / dim_products / dim_suppliers vs silver", "SELECT CONCAT((SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_customers) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.customers), '/', (SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_companies WHERE __END_AT IS NULL) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.companies), '/', (SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_products WHERE __END_AT IS NULL) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.products), '/', (SELECT COUNT(*) FROM b2b_ecommerce.gold.dim_suppliers) - (SELECT COUNT(*) FROM b2b_ecommerce.silver.suppliers))"),
    ("enrich: fact UA rows unmatched in dim_device (non-null)", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs f LEFT ANTI JOIN b2b_ecommerce.gold.dim_device d ON f.user_agent = d.user_agent WHERE f.user_agent IS NOT NULL"),
    ("enrich: fact IP rows unmatched in dim_ip_geo", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs f LEFT ANTI JOIN b2b_ecommerce.gold.dim_ip_geo d ON f.client_ip = d.client_ip WHERE f.client_ip IS NOT NULL"),
    ("nulls: fact_web_logs user_agent / referer / customer_sk (known)", "SELECT CONCAT(COUNT(*) - COUNT(user_agent), '/', COUNT(*) - COUNT(referer), '/', COUNT(*) - COUNT(customer_sk)) FROM b2b_ecommerce.gold.fact_web_logs"),
    ("nulls: fact_order_items quantity / unit_price (known 491/726, never both)", "SELECT CONCAT(COUNT(*) - COUNT(quantity), '/', COUNT(*) - COUNT(unit_price)) FROM b2b_ecommerce.gold.fact_order_items"),
    ("nulls: fact_order_items line_total / nonpositive line_total", "SELECT CONCAT(COUNT(*) - COUNT(line_total), '/', COUNT(*)) FROM b2b_ecommerce.gold.fact_order_items WHERE line_total IS NULL OR line_total <= 0"),
    ("nulls: fact_leads lead_score / estimated_order_volume / company_sk (known)", "SELECT CONCAT(COUNT(*) - COUNT(lead_score), '/', COUNT(*) - COUNT(estimated_order_volume), '/', COUNT(*) - COUNT(company_sk)) FROM b2b_ecommerce.gold.fact_leads"),
]
rows = [(label, spark.sql(q).collect()[0][0]) for label, q in checks]
display(pd.DataFrame(rows, columns=["check", "value"]))

In [0]:
# 3. KPI spot checks: recompute headline KPIs straight from gold tables.
# Expected (2026-11-19): gross $31.88M, net $28.33M, cancelled 2,386 lines/$2.487M,
# orders 11,223 (all have items), bots 140,414 (26.1%), 4xx/5xx 60,298, leads Won 161
# (144 company-matched, 140 with orders, $10,912,463.13 all-lines revenue),
# geo net: AR 24.18M / BR 2.03M / CL 1.86M / UY 0.26M, AOV $2,837.32.
import pandas as pd

checks = [
    ("KPI: GROSS revenue (all lines)", "SELECT ROUND(SUM(line_total), 2) FROM b2b_ecommerce.gold.fact_order_items"),
    ("KPI: NET revenue (excl cancelled + valid dates)", "SELECT ROUND(SUM(line_total), 2) FROM b2b_ecommerce.gold.fact_order_items WHERE status != 'cancelled' AND order_date_valid = true"),
    ("KPI: cancelled lines / revenue", "SELECT CONCAT(COUNT(*), ' / $', CAST(ROUND(SUM(line_total), 2) AS STRING)) FROM b2b_ecommerce.gold.fact_order_items WHERE status = 'cancelled'"),
    ("KPI: distinct orders in fact (silver had 11,223; 0 without items)", "SELECT COUNT(DISTINCT order_id) FROM b2b_ecommerce.gold.fact_order_items"),
    ("KPI: bot log rows / share of gross", "SELECT CONCAT(COUNT(*), ' / ', ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs), 1), '%') FROM b2b_ecommerce.gold.fact_web_logs f JOIN b2b_ecommerce.gold.dim_device d ON f.user_agent = d.user_agent WHERE d.is_bot = true"),
    ("KPI: 4xx/5xx error rows", "SELECT COUNT(*) FROM b2b_ecommerce.gold.fact_web_logs WHERE status_code >= 400"),
    ("KPI: device split at log grain (gross)", "SELECT CONCAT_WS(' | ', COLLECT_LIST(CONCAT(device_type, ':', c))) FROM (SELECT COALESCE(d.device_type, 'null-ua') device_type, COUNT(*) c FROM b2b_ecommerce.gold.fact_web_logs f LEFT JOIN b2b_ecommerce.gold.dim_device d ON f.user_agent = d.user_agent GROUP BY COALESCE(d.device_type, 'null-ua'))"),
    ("KPI: leads by funnel status", "SELECT CONCAT_WS(' | ', COLLECT_LIST(CONCAT(status, ':', c))) FROM (SELECT status, COUNT(*) c FROM b2b_ecommerce.gold.fact_leads GROUP BY status)"),
    ("KPI: Won / company-matched / matched-with-orders", "SELECT CONCAT(SUM(CASE WHEN status = 'Won' THEN 1 ELSE 0 END), ' / ', COUNT(company_sk), ' / ', (SELECT COUNT(DISTINCT l.lead_id) FROM b2b_ecommerce.gold.fact_leads l JOIN b2b_ecommerce.gold.fact_order_items f ON l.company_sk = f.company_sk)) FROM b2b_ecommerce.gold.fact_leads"),
    ("KPI: converting-lead revenue (all lines, matches silver pre-flight)", "SELECT ROUND(SUM(f.line_total), 2) FROM b2b_ecommerce.gold.fact_order_items f JOIN (SELECT DISTINCT company_sk FROM b2b_ecommerce.gold.fact_leads WHERE company_sk IS NOT NULL) l ON f.company_sk = l.company_sk"),
    ("KPI: net revenue by country (top 4)", "SELECT CONCAT_WS(' | ', COLLECT_LIST(CONCAT(country, ': ', CAST(ROUND(rev/1e6, 2) AS STRING), 'M'))) FROM (SELECT dc.country, SUM(f.line_total) rev FROM b2b_ecommerce.gold.fact_order_items f JOIN b2b_ecommerce.gold.dim_companies dc ON f.company_sk = dc.company_sk WHERE f.status != 'cancelled' AND f.order_date_valid = true GROUP BY dc.country ORDER BY rev DESC LIMIT 4)"),
    ("KPI: customers who ordered / distinct buying companies", "SELECT CONCAT(COUNT(DISTINCT customer_sk), ' / ', COUNT(DISTINCT company_sk)) FROM b2b_ecommerce.gold.fact_order_items"),
    ("KPI: AOV (valid, non-cancelled)", "SELECT ROUND(SUM(line_total) / COUNT(DISTINCT order_id), 2) FROM b2b_ecommerce.gold.fact_order_items WHERE status != 'cancelled' AND order_date_valid = true"),
]
rows = [(label, spark.sql(q).collect()[0][0]) for label, q in checks]
display(pd.DataFrame(rows, columns=["check", "value"]))